In [1]:
import sys
sys.path.append("/home/sabina/PhD/IJepa-AD/ijepa-anomaly-detection")

import os
os.chdir("/home/sabina/PhD/IJepa-AD/ijepa-anomaly-detection")

In [2]:
import torch
import os
import glob
import json
from torchvision import transforms

from src.utils.configs import *
from transformers import ViTConfig, ViTModel
from src.helper import (
    load_models_from_checkpoint,
    init_model_dinov3
)
from src.utils.anomaly_maps import *
from src.utils.preprocessing import *
from src.utils.dataset import *

/home/sabina/PhD/IJepa-AD/ijepa-anomaly-detection/.ijepa-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Set up the device
if not torch.cuda.is_available():
    device = torch.device('cpu')
else:
    device = torch.device('cuda:0')
    torch.cuda.set_device(device)

In [4]:
STORAGE_PARENT_FOLDER_PATH = "/home/sabina/PhD/BMAD"
INFERENCE_CONFIG_PATH = "configs/inference_anomaly_detection/BraTS2021_slice.yaml"

inference_anomaly_detection_config = read_inference_anomaly_detection_config(
    config_path=INFERENCE_CONFIG_PATH
)

In [5]:
subpath_list = os.getcwd().split(os.sep)
root_directory_path = os.path.join(*subpath_list[:-1])
current_dir = subpath_list[-1]

experiment_folder_path = (
    f"/{root_directory_path}/{current_dir}/"
    f"experiments_{inference_anomaly_detection_config.dataset_name}/"
    f"{inference_anomaly_detection_config.experiment_name}"
)

distillation_experiment_folder_path = (
    f"{experiment_folder_path}/checkpoints_distilled_model"
)
distillation_train_file_paths = glob.glob(os.path.join(distillation_experiment_folder_path, "*"))
for distillation_train_file_path in distillation_train_file_paths:
    if distillation_train_file_path.endswith("student_model.pth"):
        distillation_model_path = distillation_train_file_path
    elif distillation_train_file_path.endswith("config.json"):
        distillation_train_config_path = distillation_train_file_path

print("distillation_train_config_path: ", distillation_train_config_path)
print("distillation_model_path: ", distillation_model_path)
distillation_train_config = read_train_distillation_config(config_path=distillation_train_config_path)

ijepa_train_config_path = (
    f"{experiment_folder_path}/logs/config_run.yaml"
)
print(ijepa_train_config_path)
ijepa_train_config = read_train_ijepa_config(config_path=ijepa_train_config_path)
ijepa_ckpt_folder_path = (
    f"{experiment_folder_path}/checkpoints"
)
ijepa_ckpt_file_path = f"{ijepa_ckpt_folder_path}/{inference_anomaly_detection_config.experiment_name}-latest.pth.tar"
print("ijepa_ckpt_file_path", ijepa_ckpt_file_path)

save_per_patch_results_folder = (
    f"{experiment_folder_path}/results_{inference_anomaly_detection_config.split}/"
    f"image_per_patch_{inference_anomaly_detection_config.distance}_distances_"
    f"distilled_model/"
)
os.makedirs(save_per_patch_results_folder, exist_ok=True)

distillation_train_config_path:  /home/sabina/PhD/IJepa-AD/ijepa-anomaly-detection/experiments_BraTS2021_slice/enc_vits16_dinov3_pred_vitg16_ijepa_target_vits16_dinov3_cropped_random_crop_training/checkpoints_distilled_model/a5bd70b66a814550_config.json
distillation_model_path:  /home/sabina/PhD/IJepa-AD/ijepa-anomaly-detection/experiments_BraTS2021_slice/enc_vits16_dinov3_pred_vitg16_ijepa_target_vits16_dinov3_cropped_random_crop_training/checkpoints_distilled_model/a5bd70b66a814550_student_model.pth
/home/sabina/PhD/IJepa-AD/ijepa-anomaly-detection/experiments_BraTS2021_slice/enc_vits16_dinov3_pred_vitg16_ijepa_target_vits16_dinov3_cropped_random_crop_training/logs/config_run.yaml
ijepa_ckpt_file_path /home/sabina/PhD/IJepa-AD/ijepa-anomaly-detection/experiments_BraTS2021_slice/enc_vits16_dinov3_pred_vitg16_ijepa_target_vits16_dinov3_cropped_random_crop_training/checkpoints/enc_vits16_dinov3_pred_vitg16_ijepa_target_vits16_dinov3_cropped_random_crop_training-latest.pth.tar


In [6]:
# Init DinoV3/I-Jepa models: (context) encoder (DinoV3) + predictor (I-Jepa checkpoint) + target_encoder (DinoV3 - same architecture as the context encoder)
dict_models = init_model_dinov3(
    device=device,
    encoder_model_name=ijepa_train_config.encoder_model_name,
    encoder_weights_path=ijepa_train_config.encoder_weights_source,
    predictor_depth=ijepa_train_config.predictor_depth,
    encoder_embed_dim=ijepa_train_config.encoder_embed_dim,
    predictor_embed_dim=ijepa_train_config.predictor_embed_dim,
    predictor_num_patches=(ijepa_train_config.crop_size // ijepa_train_config.patch_size) * (ijepa_train_config.crop_size // ijepa_train_config.patch_size),
    predictor_num_heads=ijepa_train_config.predictor_num_heads,
    target_encoder_model_name=ijepa_train_config.target_encoder_model_name,
    target_encoder_weights_path=ijepa_train_config.target_encoder_weights_source
)
encoder = dict_models["encoder"]
predictor = dict_models["predictor"]
target_encoder = dict_models["target_encoder"]

# Load the pre-trained checkpoint (encoder + target encoder - DinoV3 VitS, predictor - IJepa checkpoint)
print(f"Loading checkpoint from: {ijepa_ckpt_file_path}")
loaded_dict_from_checkpoint = load_models_from_checkpoint(
    checkpoint_path=ijepa_ckpt_file_path,
    encoder=encoder,
    predictor=predictor,
    target_encoder=target_encoder,
    optimizer=None,
    scaler=None,
    dinov3_ckpt=True
)

target_encoder = loaded_dict_from_checkpoint["target_encoder"]

del loaded_dict_from_checkpoint

Using cache found in /home/sabina/.cache/torch/hub/facebookresearch_dinov3_main


INFO:dinov3:using base=100 for rope new
INFO:dinov3:using min_period=None for rope new
INFO:dinov3:using max_period=None for rope new
INFO:dinov3:using normalize_coords=separate for rope new
INFO:dinov3:using shift_coords=None for rope new
INFO:dinov3:using rescale_coords=2 for rope new
INFO:dinov3:using jitter_coords=None for rope new
INFO:dinov3:using dtype=fp32 for rope new
INFO:dinov3:using mlp layer as FFN
INFO:root:Correctly initialized pretrained encoder (DinoV3 checkpoint)!
INFO:root:Correctly initialized pretrained predictor (to load the weights from an IJepa checkpoint)!
INFO:dinov3:using base=100 for rope new
INFO:dinov3:using min_period=None for rope new
INFO:dinov3:using max_period=None for rope new
INFO:dinov3:using normalize_coords=separate for rope new
INFO:dinov3:using shift_coords=None for rope new
INFO:dinov3:using rescale_coords=2 for rope new
INFO:dinov3:using jitter_coords=None for rope new
INFO:dinov3:using dtype=fp32 for rope new
INFO:dinov3:using mlp layer as F

Using cache found in /home/sabina/.cache/torch/hub/facebookresearch_dinov3_main


INFO:root:Correctly loaded pretrained encoder from epoch 20 with msg: <All keys matched successfully>
INFO:root:Correctly loaded pretrained predictor from epoch 20 with msg: <All keys matched successfully>
INFO:root:Correctly loaded pretrained target encoder from epoch 20 with msg: <All keys matched successfully>
INFO:root:Correctly loaded the weights of the models from the checkpoint saved at the path: /home/sabina/PhD/IJepa-AD/ijepa-anomaly-detection/experiments_BraTS2021_slice/enc_vits16_dinov3_pred_vitg16_ijepa_target_vits16_dinov3_cropped_random_crop_training/checkpoints/enc_vits16_dinov3_pred_vitg16_ijepa_target_vits16_dinov3_cropped_random_crop_training-latest.pth.tar


In [7]:
# Loading the trained model
configuration = ViTConfig(
    hidden_size=distillation_train_config.hidden_size,
    num_hidden_layers=distillation_train_config.num_hidden_layers,
    num_attention_heads=distillation_train_config.num_attention_heads,
    intermediate_size=distillation_train_config.intermediate_size
)

student_model = ViTModel(configuration).to(device)
student_model.load_state_dict(torch.load(distillation_model_path, weights_only=True))

<All keys matched successfully>

In [8]:
# Load the evaluation dataset

EVAL_DATASET_PATH = (
    f"{STORAGE_PARENT_FOLDER_PATH}/{inference_anomaly_detection_config.dataset_name}/"
    f"{inference_anomaly_detection_config.split}"
)

image_transform = transforms.Compose([
    transforms.Resize((ijepa_train_config.crop_size, ijepa_train_config.crop_size), Image.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=ijepa_train_config.transform_mean,
        std=ijepa_train_config.transform_std
    ),
])

eval_dataset = LabelDataset(
    images_dir=EVAL_DATASET_PATH,
    dataset_name=inference_anomaly_detection_config.dataset_name,
    count_border_pixels=ijepa_train_config.count_border_pixels,
    transform=image_transform
)

eval_loader = torch.utils.data.DataLoader(
    dataset=eval_dataset,
    batch_size=1,
    shuffle=False,
    drop_last=False
)

print("Number of evaluation samples: ", len(eval_dataset))
print("Number of evaluation batches: ", len(eval_loader))

Number of evaluation samples:  3715
Number of evaluation batches:  3715


In [9]:
image_per_patch_distances = {}
for valid_batch_idx, (image, label, image_name, borders_dict) in enumerate(eval_loader):
    image = image.to(device)
    label = label.detach().cpu().item()
    image_name = image_name[0].split('.')[0]

    with torch.no_grad():
        # --- Target encoder (teacher) forward prop
        target_encoder_emb = target_encoder(image)
        # --- Student model forward prop
        student_emb = student_model(image).last_hidden_state[:, 1: , :]

        assert target_encoder_emb.shape == student_emb.shape

        # --- Compute the metrics
        # Average distance between the predicted and target patch-level representations (embeddings) for the random box
        if inference_anomaly_detection_config.distance == "l2":
            distance_per_patch = torch.linalg.vector_norm(student_emb - target_encoder_emb, ord=2, dim=2)
        elif inference_anomaly_detection_config.distance == "l1":
            distance_per_patch = torch.linalg.vector_norm(student_emb - target_encoder_emb, ord=1, dim=2)
        else:
            print("Distance metric is not correct! Set DISTANCE to one of the following options: l1, l2")
            break
        
        num_patches = distance_per_patch.shape[1]
        image_per_patch_distances[image_name] = {patch_idx: [] for patch_idx in range(num_patches)}
        for patch_idx in range(num_patches):
            image_per_patch_distances[image_name][patch_idx] = [distance_per_patch[0, patch_idx].item()]

        save_dict_path = os.path.join(
            save_per_patch_results_folder,
            image_name + "_distances_label" + "_" + str(label) + ".json"
        )
        with open(save_dict_path, "w") as f:
            json.dump(image_per_patch_distances[image_name], f)
        